# PSI-159 / CFI-G — modele 3D płomienia dla wielu test pointów (HOBJ)

Buduje **osobne pole ε(r,z) dla każdego test pointu** z prawdziwych klatek kamery
**ACME DataCam** (`.hobj`). Wystarczy dodać kolejne pary `id → zip` do słownika `TESTS`.

**Pipeline (per test):** parsowanie HOBJ → tło z ciemnych klatek → oś symetrii i pasmo
z mediany klatek „gorących" → profil radialny → **inwersja Abela** → `lut.bin` (256×128)
+ `index.json` (klatki podglądu) dla aplikacji.

**Ograniczenia:** kamera ACME (widzialna), nie INTNS; założenie symetrii osiowej;
to przybliżenie, nie pomiar wolumenu 3D.

In [ ]:
#@title 1. Zależności i Google Drive
import os, sys, json, zipfile, hashlib, re
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
IN_COLAB='google.colab' in sys.modules
DRIVE='/content/drive'
if IN_COLAB:
    from google.colab import drive; drive.mount(DRIVE)

In [ ]:
#@title 2. KONFIGURACJA — dodaj tu kolejne test pointy
TESTS = {
    '21088A1': '/content/drive/MyDrive/PSI-159/PSI-159_Raw Data_21088A1_Images.zip',
    '21088B1': '/content/drive/MyDrive/PSI-159/PSI-159_Raw Data_21088B1_Images.zip',
    '21088C1': '/content/drive/MyDrive/PSI-159/PSI-159_Raw Data_21088C1_Images.zip',
}
OUT_ROOT='/content/psi159_out'
os.makedirs(OUT_ROOT, exist_ok=True)

# jeśli pliku nie ma pod podaną ścieżką, spróbuj znaleźć go na Drive
for tid, path in list(TESTS.items()):
    if not os.path.exists(path) and IN_COLAB:
        name=os.path.basename(path)
        for root,_,files in os.walk(DRIVE):
            if name in files:
                TESTS[tid]=os.path.join(root,name); break
print(json.dumps({k:os.path.exists(v) for k,v in TESTS.items()}, indent=1))

In [ ]:
#@title 3. Funkcje: HOBJ, filtr Gaussa, Abel
def gauss(a,s):
    if s<=0: return a
    k=int(3*s)|1; x=np.arange(k)-k//2; w=np.exp(-(x*x)/(2*s*s)); w/=w.sum()
    b=np.apply_along_axis(lambda r: np.convolve(r,w,'same'),1,a)
    return np.apply_along_axis(lambda c: np.convolve(c,w,'same'),0,b)

def abel_matrix(N):
    A=np.zeros((N,N))
    for k in range(N):
        for j in range(k,N):
            A[k,j]=2*(np.sqrt(max((j+1)**2-k*k,0))-np.sqrt(max(j*j-k*k,0)))
    return A

def tms(name):
    m=re.search(r'_(\d{6})\.(\d{3})_',name)
    return (((int(m.group(1)[:2])*60+int(m.group(1)[2:4]))*60+int(m.group(1)[4:6]))*1000+int(m.group(2))) if m else 0

In [ ]:
#@title 4. build_test() — pełny pipeline dla jednego test pointu
def build_test(zip_path, out_dir, test_id):
    os.makedirs(out_dir+'/frames', exist_ok=True)
    z=zipfile.ZipFile(zip_path)
    names=sorted(n for n in z.namelist() if n.endswith('.hobj')); N=len(names)
    def load(i):
        b=z.read(names[i]); return np.frombuffer(b[-1360*1024*2:],dtype='<u2').reshape(1024,1360).astype(np.float32)
    # tło z ciemnych klatek
    probes=list(range(0,N,6)); meds={}; p99s={}
    for i in probes:
        f=load(i); meds[i]=float(np.median(f)); p99s[i]=float(np.percentile(f,99))
    dark=[i for i in probes if meds[i]<100]
    bg=np.median(np.stack([load(i) for i in dark[::2]]),axis=0) if len(dark)>=3 else np.median(np.stack([load(i) for i in probes[:5]]),axis=0)
    # klatki "gorące" -> pasmo i oś symetrii
    p99med=np.median([p99s[i] for i in probes])
    hot=[i for i in probes if p99s[i]>2*p99med and meds[i]<500] or [i for i in probes if meds[i]<500]
    bm=np.median(np.stack([load(i) for i in hot]),axis=0)
    sig=gauss(np.clip(bm-bg,0,None),3.0)
    rowsig=sig.mean(axis=1); ys=np.where(rowsig>0.4*rowsig.max())[0]
    if len(ys)<3: ys=np.array([400,900])
    y0,y1=int(ys.min()),int(ys.max())
    band=sig[y0:y1+1]; cols=band.sum(axis=0)
    cx=int((cols*np.arange(1360)).sum()/(cols.sum()+1e-9))
    R=min(cx,1360-cx); rgrid=np.arange(0,R,4); M=len(rgrid); A=abel_matrix(M)
    rows=list(range(max(0,y0-40),min(1024,y1+41),4))
    def reconstruct(i):
        s=gauss(np.clip(load(i)-bg,0,None),3.0)
        profs=np.array([0.5*(s[y,cx-rgrid]+s[y,cx+rgrid]) for y in rows])
        f=np.zeros_like(profs)
        for k in range(M-1,-1,-1):
            acc=f[:,k+1:]@A[k,k+1:] if k+1<M else np.zeros(len(rows))
            f[:,k]=(profs[:,k]-acc)/max(A[k,k],1e-9)
        eps=np.clip(f,0,None)
        return np.array(Image.fromarray(eps.astype(np.float32),mode='F').resize((128,256),Image.BILINEAR))
    idx=sorted(set(np.linspace(0,N-1,120).astype(int).tolist())); t0=tms(names[idx[0]])
    meta_frames=[]; raw=[]
    for k,i in enumerate(idx):
        f=load(i)
        Image.fromarray((np.clip(f/2000.0,0,1)*255).astype(np.uint8)).resize((256,192),Image.BILINEAR).save(f'{out_dir}/frames/f{k:03d}.jpg',quality=80)
        meta_frames.append({"i":k,"src":i,"file":f'f{k:03d}.jpg',"tMs":tms(names[i])-t0})
        raw.append(reconstruct(i) if meds.get(i,np.median(f))<500 else np.zeros((256,128),np.float32))
    gm=max(float(e.max()) for e in raw) or 1.0
    raw=[e if (e.max()>=0.01*gm) else np.zeros_like(e) for e in raw]
    lut=np.stack([np.clip(e/gm*255,0,255).astype(np.uint8) for e in raw]); lut.tofile(out_dir+'/lut.bin')
    json.dump({"dataset":"PSI-159","testpoint":test_id,"camera":"ACME DataCam (HOBJ)",
               "sourceFile":os.path.basename(zip_path),"frames":len(raw),"z":256,"r":128,
               "axisX":int(cx),"bandY":[y0,y1],"fps":12,"globalMax":gm,
               "sha256":hashlib.sha256(open(zip_path,'rb').read()).hexdigest()}, open(out_dir+'/lut-index.json','w'),indent=1)
    json.dump({"testpoint":test_id,"totalFrames":N,"sampled":len(idx),"w":1360,"h":1024,
               "displayW":256,"displayH":192,"frames":meta_frames}, open(out_dir+'/index.json','w'),indent=1)
    print(f"{test_id}: frames={N} axis={cx} band={y0}..{y1} lut={lut.nbytes}B")

In [ ]:
#@title 5. Zbuduj wszystkie test pointy
for tid, path in TESTS.items():
    if not os.path.exists(path):
        print('POMINIĘTY (brak pliku):', tid, path); continue
    build_test(path, os.path.join(OUT_ROOT, tid), tid)

In [ ]:
#@title 6. Walidacja — pierwsza klatka z płomieniem każdego testu
for tid in TESTS:
    d=os.path.join(OUT_ROOT,tid)
    if not os.path.exists(d+'/lut.bin'): continue
    m=json.load(open(d+'/lut-index.json')); lut=np.fromfile(d+'/lut.bin',np.uint8).reshape(m['frames'],256,128)
    k=int(np.argmax([np.count_nonzero(lut[i]) for i in range(lut.shape[0])]))
    fig,ax=plt.subplots(1,2,figsize=(9,3))
    ax[0].imshow(Image.open(f'{d}/frames/f{k:03d}.jpg'),cmap='gray'); ax[0].set_title(f'{tid} klatka #{k}')
    ax[1].imshow(lut[k],cmap='inferno'); ax[1].set_title('ε(r,z) — Abel')
    plt.tight_layout(); plt.show()

In [ ]:
#@title 7. Katalog tests.json + kopia na Drive
tests=[{"id":t,"label":t,"frames":json.load(open(os.path.join(OUT_ROOT,t,'lut-index.json')))["frames"]}
       for t in TESTS if os.path.exists(os.path.join(OUT_ROOT,t,'lut.bin'))]
json.dump({"dataset":"PSI-159","investigation":"ACME CFI-G","tests":tests}, open(OUT_ROOT+'/tests.json','w'), indent=1)
if IN_COLAB:
    import shutil
    dest=os.path.join(DRIVE,'MyDrive','psi159_tests'); os.makedirs(dest,exist_ok=True)
    for t in tests:
        shutil.copytree(os.path.join(OUT_ROOT,t['id']), os.path.join(dest,t['id']), dirs_exist_ok=True)
    shutil.copy(OUT_ROOT+'/tests.json', dest)
    print('skopiowano do', dest)

## Podłączenie do aplikacji
Skopiuj wynikowe katalogi do `app/app/public/psi159/` (`21088A1/`, `21088B1/`, …) razem z
`tests.json`. Aplikacja pokaże dropdown **Test** i będzie ładować `lut.bin` + `index.json`
wybranego test pointu. Możesz też otworzyć wprost `?test=21088B1#vr`.

**Uwaga:** to rekonstrukcja przybliżona z kamery ACME (widzialnej). Do czystszego modelu
potrzebne są klatki **INTNS** (wzmocniona kamera).